# Writing Your Own Algorithm

Every strategy -- built-in or yours -- implements one method:
`generate_target_weights(prices) -> weights`. Subclass `Algorithm`, return `0.0` (not `NaN`) during warm-up, and keep `sum(abs(weights)) <= 1`. That's the whole contract; the backtester takes it from there.

In [1]:
import pandas as pd
from algorithm import Algorithm


class Momentum(Algorithm):
    """Long a stock when its trailing N-day return is positive, else flat."""

    def __init__(self, ticker, lookback=20):
        self.ticker, self.lookback = ticker, lookback

    def generate_target_weights(self, prices):
        momentum = prices[self.ticker].pct_change(self.lookback)
        weight = (momentum > 0).astype(float)
        weight[momentum.isna()] = 0.0  # warm-up -> flat, not NaN
        return weight.to_frame(self.ticker)

In [2]:
from datetime import date

from data import get_prices
from backtest import Backtester, build_report

TICKER, BENCHMARK = "AAPL", "SPY"
START, END = "2018-01-01", date.today().isoformat()  # through today

prices = get_prices([TICKER], START, END)
benchmark = get_prices([BENCHMARK], START, END)[BENCHMARK]

algorithm = Momentum(TICKER, lookback=20)
result = Backtester().run(algorithm, prices, initial_capital=100_000, commission_bps=5, slippage_bps=5)
report = build_report(result, benchmark_prices=benchmark)
report.stats

{'total_return': np.float64(3.7600002548853713),
 'cagr': np.float64(0.20231237900310295),
 'annual_volatility': np.float64(0.1982846666293525),
 'sharpe': 1.0282484910543994,
 'sortino': 1.266475310502836,
 'max_drawdown': -0.2632174145719809,
 'calmar': np.float64(0.7686131988344468),
 'win_rate': 0.5014044943820225,
 'trade_count': 168,
 'benchmark_total_return': np.float64(2.1648934261118344),
 'benchmark_cagr': np.float64(0.1457409961792573)}

**Reading the numbers:** `sharpe`/`sortino` above ~1 is solid, above ~2 is excellent (return per unit of risk taken). `max_drawdown` is the worst peak-to-trough loss you'd have sat through -- closer to 0 is better. Compare `cagr` to `benchmark_cagr` to see whether the strategy actually beat just holding the benchmark.

In [3]:
for fig in report.figures:
    fig.show()